# Random Forest and AdaBoost

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

Tree ensembles combine many weak or unstable trees to improve classification performance.

**Highest-probability exam pattern:** Compare a single tree, random forest, and AdaBoost across simulation replicates and summarize accuracy.

## Assumptions, intuition, and theory

Random forests reduce variance by averaging many decorrelated trees. AdaBoost sequentially reweights difficult observations and can reduce bias, but it can be sensitive to noise.

## Python method notes and documentation

`RandomForestClassifier` uses many bootstrap-like trees with random feature selection, `AdaBoostClassifier` builds a sequential ensemble, and repeated train/test simulation shows variability.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)
- [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [plot_tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for simulation summaries.
import matplotlib.pyplot as plt  # Import plotting tools for comparison plots.
from sklearn.datasets import make_moons  # Import nonlinear classification simulator.
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier  # Import ensemble classifiers.
from sklearn.metrics import accuracy_score  # Import classification accuracy.
from sklearn.model_selection import train_test_split  # Import train/test splitting.
from sklearn.tree import DecisionTreeClassifier  # Import decision tree baseline.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
rows = []  # Create a list for simulation accuracy rows.
for rep in range(25):  # Repeat the comparison across simulated data sets.
    X, y = make_moons(n_samples=220, noise=0.28, random_state=1000 + rep)  # Simulate one nonlinear classification data set.
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=2000 + rep)  # Split this replicate into train and test sets.
    models = {  # Define the classifiers to compare.
        'tree': DecisionTreeClassifier(random_state=RANDOM_SEED),  # Add a single decision tree baseline.
        'random forest': RandomForestClassifier(n_estimators=120, random_state=RANDOM_SEED),  # Add bagged random-feature trees.
        'AdaBoost': AdaBoostClassifier(n_estimators=120, random_state=RANDOM_SEED),  # Add boosted shallow-tree ensemble.
    }  # End the model dictionary.
    for name, model in models.items():  # Loop through the candidate classifiers.
        model.fit(X_train, y_train)  # Fit the model on the replicate training data.
        accuracy = accuracy_score(y_test, model.predict(X_test))  # Compute held-out accuracy.
        rows.append({'rep': rep, 'method': name, 'accuracy': accuracy})  # Store replicate accuracy.
results = pd.DataFrame(rows)  # Convert simulation results to a DataFrame.
summary = results.groupby('method')['accuracy'].agg(['mean', 'std']).reset_index()  # Summarize accuracy by method.
display(summary)  # Display the comparison table.
box_data = [group['accuracy'].values for method, group in results.groupby('method')]  # Gather accuracy arrays for boxplots.
box_labels = [method for method, group in results.groupby('method')]  # Gather method labels for boxplots.
plt.figure(figsize=(6, 4))  # Create a comparison figure.
plt.boxplot(box_data, labels=box_labels)  # Plot accuracy distributions by method.
plt.ylabel('test accuracy')  # Label the accuracy axis.
plt.title('Tree ensembles usually reduce single-tree instability')  # Title the plot with the main message.
plt.xticks(rotation=20)  # Rotate method labels for readability.
plt.show()  # Render the boxplot.


## Exam-style problem prompt

A prompt asks whether an ensemble improves over a single tree. Run a small simulation or train/test comparison and report an accuracy table or boxplot.

## Adaptable code pattern

```python
# Step 1: Define a single tree and one or more ensemble classifiers.
# Step 2: Repeat train/test simulation or use CV.
# Step 3: Fit every method on the same training data.
# Step 4: Compute held-out accuracy for each method.
# Step 5: Summarize the accuracy distribution by method.
# Step 6: Explain random forest averaging and boosting reweighting.
```

## What to remember for the exam

Random forests mainly stabilize trees. AdaBoost builds sequentially and focuses on hard cases. Compare them by held-out accuracy, not training accuracy.